# 5-12 증권사 리포트 가져오기 — 섹션별·종목별·키워드별 리포트와 PDF

**질문 예시:** "이번 주 나온 산업 리포트 목록 줘", "SK하이닉스 최근 3개월 리포트 다 모아줘",
"반도체 관련 리포트 최신 것 PDF 내용 보여줘"

DeepSearch `["research"]`에는 증권사 리포트가 6개 섹션으로 들어 있습니다:
`company`(기업), `industry`(산업), `market`(시황), `strategy`(투자전략), `economy`(경제), `bond`(채권).
문서마다 증권사(`publisher`)와 PDF 링크가 있습니다. 목표주가·의견을 엮은 분석은 [2-11 증권사 리포트 흐름](../2_disclosure_news/11_analyst_reports.ipynb)을 보세요.

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/beaten-by-the-market/deepsearch/blob/main/scenarios/5_data_lookup/12_research.ipynb)

## 0. 준비
Colab 왼쪽 🔑(Secrets)에 `DEEPSEARCH_API_KEY`를 등록하고 이 노트북의 액세스를 켠 뒤 아래 두 셀을 실행하세요. 두 번째 셀은 모든 시나리오에 공통인 헬퍼입니다(접어 두어도 됩니다).

In [1]:
# Colab에서만: 한글 차트 폰트 설치 (로컬에서는 건너뜀)
import sys
if "google.colab" in sys.modules:
    !pip -q install koreanize-matplotlib

In [2]:
# ── DeepSearch 헬퍼 (모든 시나리오 노트북 공통) ─────────────────────────────
# Colab: 왼쪽 🔑(Secrets)에 이름 DEEPSEARCH_API_KEY 로 키를 넣고 "노트북 액세스"를 켜세요.
#        키는 "아이디:비밀번호" 형식이든 base64 문자열이든 됩니다. 키 값은 출력하지 않습니다.
# 로컬: 환경변수 DEEPSEARCH_API_KEY / API_KEY 또는 상위 폴더의 .env 를 읽습니다.
import base64, json, os, sys, time
from datetime import date, datetime, timedelta

import pandas as pd
import requests

IN_COLAB = "google.colab" in sys.modules
API_URL = "https://api.deepsearch.com/v1/compute"

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)


def _norm_key(k):
    k = k.strip().strip('"').strip("'")
    return base64.b64encode(k.encode()).decode() if ":" in k else k


def _find_key():
    if IN_COLAB:
        try:
            from google.colab import userdata
            k = userdata.get("DEEPSEARCH_API_KEY")
            if k:
                return _norm_key(k)
        except Exception:
            pass
    for name in ("DEEPSEARCH_API_KEY", "API_KEY"):
        if os.environ.get(name):
            return _norm_key(os.environ[name])
    d = os.path.abspath(os.getcwd())
    while True:
        p = os.path.join(d, ".env")
        if os.path.isfile(p):
            for line in open(p, encoding="utf-8-sig"):
                if "=" in line and not line.strip().startswith("#"):
                    k, v = line.split("=", 1)
                    if k.strip() in ("DEEPSEARCH_API_KEY", "API_KEY") and v.strip():
                        return _norm_key(v)
        if os.path.dirname(d) == d:
            break
        d = os.path.dirname(d)
    from getpass import getpass
    return _norm_key(getpass("DeepSearch API 키: "))


_KEY = _find_key()


class DSError(Exception):
    pass


def ds(query, timeout=180, retries=3):
    """compute 호출 → 응답 JSON. 서버가 돌려준 오류(data.exceptions)는 DSError."""
    last = None
    for i in range(retries):
        try:
            r = requests.post(API_URL, json={"input": query}, timeout=timeout,
                              headers={"Authorization": "Basic " + _KEY})
            body = r.json()
        except Exception as e:  # 네트워크·일시 오류만 재시도
            last = e
            time.sleep(3 * (i + 1))
            continue
        exc = (body.get("data") or {}).get("exceptions") or []
        if exc or r.status_code >= 400:
            msg = "; ".join(f"{x.get('class', '')}: {' '.join(map(str, x.get('args') or [x.get('message', '')]))}"
                            if isinstance(x, dict) else str(x) for x in exc)
            raise DSError(f"{r.status_code} {msg or body}"[:500])
        return body
    raise DSError(f"요청 실패: {last}")


def _pod(body):
    pods = (body.get("data") or {}).get("pods") or []
    return pods[1] if len(pods) >= 2 else {}


def ds_df(query, **kw):
    """DataFrame 결과(재무·시세·경제지표 등) → pandas DataFrame. 결과가 없으면 빈 DataFrame."""
    c = _pod(ds(query, **kw)).get("content") or {}
    data = c.get("data") if isinstance(c, dict) else None
    if not isinstance(data, dict) or not data:
        return pd.DataFrame()
    cols = [x for x in (c.get("columns") or data.keys()) if x in data]
    df = pd.DataFrame({x: data[x] for x in cols})
    for x in ("date", "created_at"):
        if x in df and df[x].dtype == object:
            df[x] = pd.to_datetime(df[x].astype(str).str[:19], errors="coerce")
    return df


def _polarity(x):
    """문서 긍부정: polarity.name, 없으면 customized_ml.kis 모델 값(최근 뉴스는 절반가량이 이쪽에만 있음)."""
    p = x.get("polarity")
    if isinstance(p, dict) and p.get("name"):
        return p["name"]
    k = (((x.get("customized_ml") or {}).get("kis") or {}).get("ip_multitask") or {}).get("polarity") or {}
    return k.get("name")


def doc_search(category, section, query, date_from, date_to, count=100, pages=1):
    """DocumentSearch → (총건수, 문서 DataFrame). category: news/research/company/patent. 기간은 1년 이내.
    종목 지정은 securities.symbol:005930 (접두어 없는 6자리)이 뉴스·리서치·공시 모두에서 가장 정확합니다.
    securities.symbol:KRX:005930 은 키워드 검색으로 바뀌어 건수가 줄고, securities.name: 은 리서치(공식명
    '에스케이하이닉스')·공시(DART명 '현대자동차')에서 놓칩니다(2026-11-25 확인). 여러 종목: securities.symbol:(005930 or 000660)"""
    rows, total = [], 0
    for page in range(1, pages + 1):
        q = (f'DocumentSearch(["{category}"],{json.dumps(section, ensure_ascii=False)},'
             f'{json.dumps(query, ensure_ascii=False)},count={count},page={page},'
             f'date_from={date_from},date_to={date_to})')
        c = _pod(ds(q)).get("content") or {}
        d = c.get("data") if isinstance(c.get("data"), dict) else c
        total = d.get("total_matches") or 0
        docs = d.get("docs") or []
        for x in docs:
            secs = x.get("securities") or []
            rows.append({
                "created_at": pd.to_datetime(str(x.get("created_at"))[:19], errors="coerce"),
                "publisher": x.get("publisher"),
                "title": " ".join(str(x.get("title") or "").split()),
                "stocks": ", ".join(s.get("name", "") for s in secs),
                "symbols": ", ".join(s.get("symbol", "") for s in secs),
                "polarity": _polarity(x),
                "url": x.get("content_url"),
            })
        if len(docs) < count:
            break
    cols = ["created_at", "publisher", "title", "stocks", "symbols", "polarity", "url"]
    return total, pd.DataFrame(rows, columns=cols)


def doc_trends(category, section, query, date_from, date_to, interval="1d"):
    """DocumentTrends → DataFrame(date, count)."""
    q = (f'DocumentTrends(["{category}"],{json.dumps(section, ensure_ascii=False)},'
         f'{json.dumps(query, ensure_ascii=False)},interval="{interval}",'
         f'date_from={date_from},date_to={date_to})')
    c = _pod(ds(q)).get("content") or {}
    d = c.get("data") if isinstance(c.get("data"), dict) else c
    trends = d.get("trends") or []
    b = trends[0].get("buckets", []) if trends else []
    df = pd.DataFrame([{"date": x.get("key_as_string") or x.get("key"), "count": x.get("count", 0)} for x in b])
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"].astype(str).str[:19], errors="coerce")
    return df


def stock_prices(entities, date_from, date_to, columns=None):
    """종목 목록의 일별 시세. entities 예: ["KRX:005930", "KRX:000660"] 또는 "KRX:005930".
    가격은 수정주가, 거래량·거래대금·시가총액은 당일 실제 값. change_rate 는 비율(0.01 = 1%)."""
    ents = [entities] if isinstance(entities, str) else list(entities)
    cols = columns or ["open", "high", "low", "close", "volume", "value", "change_rate", "market_cap"]
    q = (f'GetStockPrices([{", ".join(ents)}], columns={json.dumps(cols)}, '
         f'date_from={date_from}, date_to={date_to})')
    return ds_df(q)


_UNIVERSE = None


def universe():
    """현재 상장 종목 기본정보(GetStockSymbols 전 종목): symbol, name, market, industry_name 등.
    company_type_l1: 1 일반법인, 2 공기업(한국전력·기업은행 등 8종목), 8 ETF·ETN 등."""
    global _UNIVERSE
    if _UNIVERSE is None:
        u = ds_df("GetStockSymbols()")
        _UNIVERSE = u.rename(columns={"entity_name": "name"})[
            ["symbol", "name", "market", "company_type_l1", "industry_id", "industry_name", "date_listed",
             "fiscal_year_end", "conglomerate_id"]]
    return _UNIVERSE.copy()


def all_stocks(day, common_only=True, columns=None):
    """day(또는 그 이전 마지막 거래일)의 전 종목 하루치 시세 + 종목 정보. 한 번 호출로 4,000여 종목.
    common_only=True 이면 일반법인·공기업 보통주만(ETF·ETN·우선주 제외)."""
    day = pd.Timestamp(day).date()
    cols = columns or ["open", "high", "low", "close", "volume", "value", "change_rate", "market_cap"]
    snap = ds_df(f'GetStockPrices(columns={json.dumps(cols)}, '
                 f'date_from={day - timedelta(days=7)}, date_to={day})')
    df = snap.merge(universe().drop(columns=["date"], errors="ignore"), on="symbol", how="left")
    if common_only:
        df = df[df["company_type_l1"].isin(["1", "2"]) & df["symbol"].str.endswith("0")]
    df = df[df["market"].isin(["KOSPI", "KOSDAQ", "KONEX"])]
    if "change_rate" in df:
        df["change_pct"] = df["change_rate"] * 100
    return df.reset_index(drop=True)


def index_prices(date_from, date_to):
    """지수 11종(코스피·코스닥 계열) 일별. change_rate 는 퍼센트(1.94 = 1.94%).
    지수의 value(거래대금)는 백만원, volume(거래량)은 천주 단위로 보입니다(코스닥 2026-11-23 value 4,673,819 ≈ 4.7조)."""
    # 지수를 지정하지 않으면 최근 약 1,300행(≈118거래일)만 오고 앞 기간이 조용히 잘린다 → 140일씩 나눠 받는다
    start, end = pd.Timestamp(date_from), pd.Timestamp(date_to)
    parts = []
    while start <= end:
        stop = min(start + pd.Timedelta(days=139), end)
        parts.append(ds_df(f"GetMarketIndexes(date_from={start.date()}, date_to={stop.date()})"))
        start = stop + pd.Timedelta(days=1)
    df = pd.concat([x for x in parts if not x.empty], ignore_index=True) if parts else pd.DataFrame()
    return df.drop_duplicates(["date", "symbol"]).sort_values(["date", "symbol"]).reset_index(drop=True) if not df.empty else df


def fred(series, date_from, date_to):
    """FRED 시리즈 → 날짜 × 시리즈 넓은 표. 예: fred(["DGS10", "DEXKOUS"], "2026-10-01", "2026-11-24")"""
    ids = ", ".join(json.dumps(s) for s in series)
    df = ds_df(f"QueryFredSeriesData([{ids}], date_from={date_from}, date_to={date_to})")
    if df.empty:
        return df
    val = [c for c in df.columns if c not in ("entity_id", "date")][0]
    df["series"] = df["entity_id"].str.replace("FRED:", "", regex=False)
    wide = df.pivot_table(index="date", columns="series", values=val).sort_index()
    return wide[[s for s in series if s in wide.columns]]  # 요청한 순서대로


def krw(v, digits=1):
    """원 → '138.0조' / '5,027억'."""
    if v is None or pd.isna(v):
        return "-"
    a = abs(v)
    if a >= 1e12:
        return f"{v / 1e12:.{digits}f}조"
    if a >= 1e8:
        return f"{v / 1e8:,.0f}억"
    if a >= 1e4:
        return f"{v / 1e4:,.0f}만"
    return f"{v:,.0f}"


PRICE_NOTICE = ("※ 주가는 수정주가입니다(분할·무상증자·유상증자 소급 반영, 배당 미반영). "
                "거래량·거래대금·시가총액은 당일 실제 값이며 KRX 기준입니다(NXT 미포함).")

# 한글 차트 폰트
import matplotlib.pyplot as plt
try:
    import koreanize_matplotlib  # noqa: F401  (Colab: 위 셀에서 설치)
except ImportError:
    if sys.platform.startswith("win"):
        plt.rcParams["font.family"] = "Malgun Gothic"
    elif sys.platform == "darwin":
        plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

print("DeepSearch 준비 완료", "(Colab)" if IN_COLAB else "(로컬)")

DeepSearch 준비 완료 (로컬)


In [3]:
ASOF = "2026-11-24"
STOCK = "000660"       # SK하이닉스
KEYWORD = "반도체"

if IN_COLAB:
    !pip -q install pymupdf

SECTIONS = {"기업": "company", "산업": "industry", "시황": "market", "투자전략": "strategy", "경제": "economy", "채권": "bond"}

## 1. 섹션별 리포트 수 (최근 4주)
건수는 문서를 받지 않고 `total_matches`만 봅니다(섹션당 1회 호출).

In [4]:
d4w = (pd.Timestamp(ASOF) - pd.Timedelta(days=27)).date()
cnt = {}
for name, sec in SECTIONS.items():
    t, _ = doc_search("research", [sec], "*", d4w, ASOF, count=1)
    cnt[name] = t
pd.Series(cnt, name="리포트 수(4주)").to_frame()

,리포트 수(4주)
기업,1268
산업,437
시황,263
투자전략,346
경제,52
채권,33


## 2. 증권사별 리포트 수 (최근 4주, 전 섹션)
집계는 `publisher:N`으로 합니다(`publisher.raw:N`은 집계에서 서버 오류 — 검색 쿼리 안에서는 `publisher.raw:('삼성증권')`처럼 씁니다).

In [5]:
c = ds(f'DocumentAggregation(["research"],[],"*","publisher:30",date_from={d4w},date_to={ASOF})')
pubs = pd.DataFrame(c["data"]["pods"][1]["content"]["data"])[["key", "count"]]
pubs.columns = ["증권사", "리포트 수"]
print(f"4주간 리포트를 낸 증권사 {len(pubs)}곳")
pubs.head(15)

4주간 리포트를 낸 증권사 16곳


,증권사,리포트 수
0,신한투자증권,432
1,삼성증권,428
2,미래에셋증권,281
3,한국투자증권,252
4,SK증권,156
5,유진투자증권,150
6,유안타증권,125
7,LS증권,121
8,메리츠증권,120
9,IBK투자증권,117


## 3. 종목별 — SK하이닉스 최근 3개월 기업 리포트
종목은 `securities.symbol:6자리`로 지정합니다. 리서치의 회사 이름은 공식명('에스케이하이닉스')이라
`securities.name:SK하이닉스`로는 일부만 걸립니다.

In [6]:
d3m = (pd.Timestamp(ASOF) - pd.DateOffset(months=3)).date()
t, rep = doc_search("research", ["company"], f"securities.symbol:{STOCK}", d3m, ASOF, count=100)
print(f"전체 {t}건 | 증권사별:", rep.publisher.value_counts().to_dict())
rep[["created_at", "publisher", "title", "url"]].head(10)

전체 15건 | 증권사별: {'미래에셋대우': 4, '신한금융투자': 2, 'IBK투자증권': 2, '한화투자증권': 2, 'iM증권': 1, 'SK증권': 1, '삼성증권': 1, '메리츠종금증권': 1, '대신증권': 1}


,created_at,publisher,title,url
0,2026-11-04 11:00:09,iM증권,HBM 하나로 시가총액 설명,https://ddi-cdn.deepsearch.com/research/company/2026/11/...
1,2026-10-29 10:36:55,IBK투자증권,실적은 우려를 불식시킬 것,https://ddi-cdn.deepsearch.com/research/company/2026/10/...
2,2026-10-29 10:36:55,SK증권,여전히 강력하다,https://ddi-cdn.deepsearch.com/research/company/2026/10/...
3,2026-10-29 09:00:00,미래에셋대우,SK하이닉스 (000660/매수) 주주환원보다 유의미한 장기수요 전망,https://ddi-cdn.deepsearch.com/research/company/2026/10/...
4,2026-10-29 09:00:00,신한금융투자,SK하이닉스; 거를 타선이 없었던 실적,https://ddi-cdn.deepsearch.com/research/company/2026/10/...
5,2026-10-29 08:12:53,한화투자증권,"높아지는 이익의 눈높이, 달라지는 메모리 사이클",https://ddi-cdn.deepsearch.com/research/company/2026/10/...
6,2026-10-29 07:50:42,삼성증권,"SK하이닉스(000660/BUY): 반복되는 의심, 반복되는 수익 기회",https://ddi-cdn.deepsearch.com/research/company/2026/10/...
7,2026-10-20 09:07:25,한화투자증권,"SK하이닉스 솔리다임 상장, 성장과 주주가치를 잇는 조건",https://ddi-cdn.deepsearch.com/research/company/2026/10/...
8,2026-10-12 09:22:32,메리츠종금증권,3Q26 OP 80.1조원 실적 서프라이즈 예상,https://ddi-cdn.deepsearch.com/research/company/2026/10/...
9,2026-10-07 14:33:34,IBK투자증권,실적에 수급을 더한다,https://ddi-cdn.deepsearch.com/research/company/2026/10/...


## 4. 키워드 — 산업 리포트 중 제목에 '반도체'

In [7]:
d2w = (pd.Timestamp(ASOF) - pd.Timedelta(days=13)).date()
t, ind = doc_search("research", ["industry"], f"title:{KEYWORD}", d2w, ASOF, count=100)
print(f"최근 2주 산업 리포트 중 제목에 '{KEYWORD}' {t}건")
ind[["created_at", "publisher", "title"]].head(10)

최근 2주 산업 리포트 중 제목에 '반도체' 8건


,created_at,publisher,title
0,2026-11-24 09:00:00,미래에셋대우,"[2027 전망] 중국 반도체 군계일학(群鷄一鶴), 차별화되는 이익"
1,2026-11-23 09:44:15,SK증권,"2027년 반도체 연간전망: 증명, 그리고 밸류"
2,2026-11-23 09:44:15,SK증권,2027년 반도체 소부장 연간전망: AI 실적 랠리
3,2026-11-20 09:00:00,미래에셋대우,"반도체 주가 변동성 과격, 업황은 평온"
4,2026-11-18 09:00:00,미래에셋대우,반도체 엔비디아 FY27 3Q 실적 퀵 리뷰
5,2026-11-17 09:00:00,미래에셋대우,"반도체 일부 수요 조정, 떠날 정도는 아니다"
6,2026-11-12 09:00:00,미래에셋대우,반도체 엔비디아 이슈 코멘트
7,2026-11-11 09:00:00,신한금융투자,반도체; 노이즈 캔슬링


## 5. PDF 읽기 — 최신 기업 리포트 첫 장
기업 리포트 첫 장에는 보통 투자의견·목표주가·요약이 있습니다.

In [8]:
import fitz  # PyMuPDF

def pdf_text(url, start=1, pages=1, limit=1500):
    r = requests.get(url, timeout=60); r.raise_for_status()
    doc = fitz.open(stream=r.content, filetype="pdf")
    idx = range(start - 1, min(start - 1 + pages, doc.page_count))
    text = "\n".join(doc[i].get_text() for i in idx)
    print(f"{doc.page_count}쪽 PDF, {len(text):,}자")
    return " ".join(text.split())[:limit]

latest = rep.sort_values("created_at").iloc[-1]
print(latest.created_at, latest.publisher, latest.title)
txt = pdf_text(latest.url)
print(txt)

2026-11-04 11:00:09 iM증권 HBM 하나로 시가총액 설명


12쪽 PDF, 2,532자
반도체(Overweight): Stress Test 90 SK하이닉스 (000660) 2026.11.04 HBM 하나로 시가총액 설명 [반도체/전기전자] 고의영 2122-9179 ey.ko@imfnsec.com 업황 우려에 고유 할인까지 겹친 과도한 하락 투자의견 Buy, 목표주가 230만원으로 커버리지를 개시한다. ‘27년 BPS에 목표 P/B 2.3배를 적용했다[표1]. 이는 상장 이후 평균 P/B의 +2표준편차에 해당한다. ‘26~’28년 ROE는 각각 102%, 66%, 47%로, ‘26년 정점 이후에도 과거 업사이클 고점보다 높은 수준이 유지될 전망이다. 한편, 이익 지속성에 대한 믿음이 공고해져 시장의 시야가 ‘28년으로 향할 경우 ‘28년 BPS 기준 목표주가는 360만원까지 열릴 수 있다[표2]. 동사 주가는 ‘26년 6월 고점 대비 10월 저점 기준 57% 하락해 과거 다운사이클 평균 하락폭인 -51%를 넘어섰고, 과거 8~13개월에 걸쳐 진행됐던 조정이 이번에는 4개월 만에 나타났다. 시장이 ‘28년 이익 급감을 상정하고 있는 것이다. 그러나 당사는 아래와 같은 투자포인트와 향후 주가 흐름을 제시한다. ‘28년 HBM 가치, 시가총액의 1.3배 HBM 비중이 높아질수록 동사의 이익 변동성은 낮아질 것이다. 특히 cHBM으로 갈수록 고객별 수주생산 제품으로 현물시장과 유통 재고가 없고, 생산에 앞서 물량과 가격이 결정된다. 동사는 이 전환이 업계에서 가장 선두권이다. 당사는 동사 HBM 매출이 ‘26년 43조원에서 ‘28년 183조원으로 증가할 것으로 전망한다. 만약, 동사의 ‘26~’28년 HBM 영업이익에 시기별 TSMC P/E(21배, 15배, 12배)를 적용하면 HBM 가치는 동사 시가총액의 47%, 93%, 135%를 설명한다[표3]. ‘27년에 이미 HBM 하나만으로 현재 시가총액이 설명된다는 의미다. 시장이 통상적으로 받아들이는 멀티플 10배를 타겟팅해도 결과는 크게 다르지 않다. 150만원에 사면 ‘3

In [9]:
# 첫 장에서 목표주가·투자의견 문구 찾기 (리포트 양식마다 달라 정규식은 참고용)
import re
PATTERNS = {"목표주가": r"목표주가[^0-9]{0,15}([\d,]+\s*(?:만원|원))", "투자의견": r"투자의견\s*([A-Za-z가-힣]+)"}
for name, pat in PATTERNS.items():
    m = re.search(pat, txt)
    print(name, "→", m.group(1) if m else "못 찾음")

목표주가 → 230만원
투자의견 → Buy


## 결과 읽는 법과 한계
- **증권사 커버리지가 제한적입니다.** 16개 기관(4주 기준) 리포트만 들어 있고 KB·NH·키움 등은 없습니다.
  일부 증권사는 옛 이름('미래에셋대우', '신한금융투자', '메리츠종금증권')으로 들어 있어 증권사별로 묶을 때 주의하세요.
- **목표주가·의견은 구조화된 값이 따로 있습니다**(`SearchTargetPrices`) — PDF 정규식보다 그쪽이 정확합니다(2-11).
- PDF는 DeepSearch CDN 링크라 키 없이 받을 수 있습니다. 재배포에는 각 증권사 이용 조건이 적용됩니다.